# Spatial-block CV robustness check

**Phase 2 task 3** — replicates Exp 0 (per-country 5-fold CV) under spatially blocked splits to address RSE reviewer concerns about spatial autocorrelation.

**Method:** for each country, partition parcels into a 5×5 grid of spatial blocks based on (lon, lat) percentiles, then use `GroupKFold` so train and test parcels never come from the same block. Compare against the original random `StratifiedKFold`.

**Outputs:**
- `Manuscript/Paper_Alpha_Earth/exp6_borrowing_strategies/tables/spatial_block_cv_per_country.csv`
- `Manuscript/Paper_Alpha_Earth/exp6_borrowing_strategies/figures/main/fig12_spatial_vs_random_cv.{pdf,png}`

**Runtime estimates (Colab CPU):**
- All 11 countries × 3 features × 10 folds = 330 LightGBM fits → ~3 hours
- 4 representative countries × 2 features × 10 folds = 80 fits → ~30 minutes (recommended for first run)
- Configure with `COUNTRIES_TO_RUN` and `FEATURES_TO_RUN` below.

## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

candidates = [
    Path('/content/drive/MyDrive/Alpha_Crops'),
    Path('/content/drive/Shareddrives/Alpha_Crops'),
    Path('/content/drive/MyDrive/Research/Alpha_Crops'),
]
PROJECT_ROOT = next((c for c in candidates if c.exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Edit PROJECT_ROOT manually if shortcut is elsewhere.')
print(f'Project root: {PROJECT_ROOT}')

FEATURES_DIR = PROJECT_ROOT / 'data' / 'eu_processed' / 'features'
TABLES_DIR = PROJECT_ROOT / 'Manuscript' / 'Paper_Alpha_Earth' / 'exp6_borrowing_strategies' / 'tables'
FIG_DIR = PROJECT_ROOT / 'Manuscript' / 'Paper_Alpha_Earth' / 'exp6_borrowing_strategies' / 'figures' / 'main'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Configure scope (edit these to control runtime)

In [ ]:
# All 11 main countries from the paper:
ALL_COUNTRIES = ['AT', 'BE', 'DE', 'EE', 'LT', 'LV', 'PT', 'RO', 'SE', 'SI', 'SK']

# Pick subset to run. For first pass on Colab, use a representative 4:
#   DE (Atlantic, hub), PT (Mediterranean, hardest target),
#   EE (Nordic-Baltic), SK (Continental).
# Set to ALL_COUNTRIES for the full table.
COUNTRIES_TO_RUN = ['DE', 'PT', 'EE', 'SK']           # ~30 min CPU
# COUNTRIES_TO_RUN = ALL_COUNTRIES                       # ~3 hr CPU

FEATURES_TO_RUN = ['ae', 's2']                          # add 'fusion' if budget allows
GRID_RESOLUTION = 5     # 5x5 = up to 25 spatial blocks per country
N_FOLDS = 5
SEED = 42
MAX_PARCELS_PER_COUNTRY = 50000   # subsample if a country has more (matches paper Exp 0)

print(f'Will run: {len(COUNTRIES_TO_RUN)} countries x {len(FEATURES_TO_RUN)} features '
      f'x 2 CV strategies x {N_FOLDS} folds = '
      f'{len(COUNTRIES_TO_RUN) * len(FEATURES_TO_RUN) * 2 * N_FOLDS} model fits')

## 3. Install dependencies

In [ ]:
!pip install -q lightgbm pyarrow

## 4. Load features (combining train + val + test)

In [ ]:
# 把 features 从 Drive 拷到 Colab 本地 SSD
# 文件级拷贝 + 每文件进度日志：防止 websocket 因长时间无输出掉线
import shutil, time

LOCAL = Path('/content/local_features')
LOCAL.mkdir(exist_ok=True)

t0_total = time.time()
files_copied = 0
for feat in FEATURES_TO_RUN:        # 只拷需要的（默认 ae+s2；fusion 自动跳过）
    (LOCAL / feat).mkdir(parents=True, exist_ok=True)
    for split in ['train', 'val', 'test']:
        src = FEATURES_DIR / feat / f'features_{split}.parquet'
        dst = LOCAL / feat / f'features_{split}.parquet'
        if dst.exists() and dst.stat().st_size > 0:
            print(f'  ✓ {feat}/{split}: cached ({dst.stat().st_size/1e6:.0f} MB)')
            continue
        if not src.exists():
            print(f'  - {feat}/{split}: not in Drive, skip')
            continue
        size_mb = src.stat().st_size / 1e6
        t0 = time.time()
        print(f'  → {feat}/{split} ({size_mb:.0f} MB)... ', end='', flush=True)
        shutil.copy(src, dst)
        elapsed = time.time() - t0
        print(f'done in {elapsed:.0f}s ({size_mb/max(elapsed,0.1):.1f} MB/s)')
        files_copied += 1

FEATURES_DIR = LOCAL
print(f'\n✅ Local cache ready ({files_copied} new + {(files_copied==0)*1} cached)'
      f' in {(time.time()-t0_total)/60:.1f} min')
print(f'FEATURES_DIR now: {FEATURES_DIR}')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

META_COLS = {
    'parcel_id', 'country', 'climate_zone', 'crop',
    'log_area', 'longitude', 'latitude', 'area_ha',
    'EC_hcat_n', 'split', 'size_class',
    'label_tier_a', 'label_tier_b', 'label_tier_c',
    'label_tier_a_encoded', 'label_tier_b_encoded', 'label_tier_c_encoded',
}

# Strict 2021 alignment: exclude DE_BB (uses 2023 LPIS, not aligned with the
# paper's main-analysis methodology in Section 3.1)
EXCLUDE_PARCEL_PREFIXES = ('DE_BB_',)

import pyarrow.parquet as pq

def load_country(feature_set, country):
    parts = []
    for split in ['train', 'val', 'test']:
        f = FEATURES_DIR / feature_set / f'features_{split}.parquet'
        if not f.exists():
            continue
        try:
            tbl = pq.read_table(str(f), filters=[('country', '=', country)])
            if len(tbl) > 0:
                parts.append(tbl.to_pandas())
        except Exception as e:
            print(f'  warn {f.name}: {e}')
    if not parts:
        return None
    df = pd.concat(parts, ignore_index=True)

    # Strict 2021 alignment: drop DE_BB parcels (only relevant for country=='DE')
    if country == 'DE':
        pid = df['parcel_id'].astype(str)
        bb_mask = pid.str.startswith(EXCLUDE_PARCEL_PREFIXES)
        n_dropped = int(bb_mask.sum())
        if n_dropped > 0:
            df = df[~bb_mask.values].copy()
            print(f'  [strict 2021] dropped {n_dropped:,} DE_BB parcels (2023 LPIS)')

    if len(df) > MAX_PARCELS_PER_COUNTRY:
        df = df.groupby('label_tier_b', group_keys=False).apply(
            lambda g: g.sample(min(len(g),
                                    int(MAX_PARCELS_PER_COUNTRY * len(g) / len(df))),
                                random_state=SEED)
        ).reset_index(drop=True)
    return df


# quick sanity check — DE should now be ~99,988 (capped to MAX_PARCELS_PER_COUNTRY)
test_df = load_country('ae', 'DE')
print(f'DE / ae: {test_df.shape}, classes: {test_df["label_tier_b"].nunique()}')

## 5. Spatial-block CV core

In [ ]:
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

LGBM_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
    verbose=-1,
)

def assign_spatial_blocks(df, n_grid=GRID_RESOLUTION):
    """5x5 grid of blocks based on lon/lat percentiles. Returns block_id per parcel."""
    lon = df['longitude'].values
    lat = df['latitude'].values
    lon_bins = np.percentile(lon, np.linspace(0, 100, n_grid + 1))
    lat_bins = np.percentile(lat, np.linspace(0, 100, n_grid + 1))
    lon_bins[-1] += 1e-6
    lat_bins[-1] += 1e-6
    return (np.digitize(lon, lon_bins) - 1) * n_grid + (np.digitize(lat, lat_bins) - 1)

def evaluate_cv(X, y, groups, cv_strategy):
    if cv_strategy == 'random':
        kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        splits = kf.split(X, y)
    elif cv_strategy == 'spatial_block':
        kf = GroupKFold(n_splits=N_FOLDS)
        splits = kf.split(X, y, groups)
    bas, f1s = [], []
    for tr, te in splits:
        clf = lgb.LGBMClassifier(**LGBM_PARAMS)
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        bas.append(balanced_accuracy_score(y[te], pred))
        f1s.append(f1_score(y[te], pred, average='macro'))
    return float(np.mean(bas)), float(np.std(bas)), float(np.mean(f1s)), float(np.std(f1s))

## 6. Run the full sweep (this is the long-running cell)

In [ ]:
import time, gc

# Resumable: 已完成的 (country, feature) 直接跳过，避免重跑
csv_path = TABLES_DIR / 'spatial_block_cv_per_country.csv'

if csv_path.exists() and csv_path.stat().st_size > 0:
    prior = pd.read_csv(csv_path)

    # ★ One-time migration: drop DE rows computed BEFORE the BB filter was
    #   activated. They lack the 'bb_excluded' marker column. Idempotent —
    #   subsequent runs see bb_excluded=True on new DE rows and skip this step.
    if 'bb_excluded' not in prior.columns:
        prior['bb_excluded'] = False
    de_stale_mask = (prior['country'] == 'DE') & (~prior['bb_excluded'])
    n_stale = int(de_stale_mask.sum())
    if n_stale > 0:
        print(f'⚠️  Purging {n_stale} stale DE rows (computed before BB filter '
              f'activation); they will be re-run with strict 2021 alignment.')
        prior = prior[~de_stale_mask].reset_index(drop=True)
        prior.to_csv(csv_path, index=False)

    rows = prior.to_dict('records')
    done = {(r['country'], r['feature']) for r in rows}
    print(f'Resuming from CSV: {len(done)} combos already done')
else:
    rows = []
    done = set()


def evaluate_cv_verbose(X, y, groups, cv_strategy, label):
    """每折结束打一行——保证 websocket 永远不静默超过 1-2 分钟。"""
    if cv_strategy == 'random':
        kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        splits = list(kf.split(X, y))
    else:
        kf = GroupKFold(n_splits=N_FOLDS)
        splits = list(kf.split(X, y, groups))

    bas, f1s = [], []
    for fold_i, (tr, te) in enumerate(splits, 1):
        t_f = time.time()
        clf = lgb.LGBMClassifier(**LGBM_PARAMS)
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        ba = balanced_accuracy_score(y[te], pred)
        f1 = f1_score(y[te], pred, average='macro')
        bas.append(ba); f1s.append(f1)
        print(f'    {label} fold {fold_i}/{N_FOLDS}: BA={ba:.3f}  ({time.time()-t_f:.0f}s)',
              flush=True)
        del clf, pred
        gc.collect()
    return float(np.mean(bas)), float(np.std(bas)), float(np.mean(f1s)), float(np.std(f1s))


for feat_name in FEATURES_TO_RUN:
    for country in COUNTRIES_TO_RUN:
        if (country, feat_name) in done:
            print(f'\n=== {country} / {feat_name} === (skip — already in CSV)', flush=True)
            continue

        t0 = time.time()
        print(f'\n=== {country} / {feat_name} ===', flush=True)

        df = load_country(feat_name, country)
        if df is None or len(df) < 100:
            print(f'  skip (no data)')
            continue

        feat_cols = [c for c in df.columns if c not in META_COLS]
        X = df[feat_cols].values.astype(np.float32)
        y = LabelEncoder().fit_transform(df['label_tier_b'].astype(str).values)
        blocks = assign_spatial_blocks(df)
        n_blocks = pd.Series(blocks).nunique()
        print(f'  parcels: {len(df):,}, classes: {len(np.unique(y))}, '
              f'blocks: {n_blocks}', flush=True)

        ba_r, std_r, f1_r, _ = evaluate_cv_verbose(X, y, blocks, 'random', 'random ')
        ba_s, std_s, f1_s, _ = evaluate_cv_verbose(X, y, blocks, 'spatial_block', 'spatial')
        infl = (ba_r - ba_s) * 100

        print(f'  Random  : BA={ba_r:.3f} ± {std_r:.3f}  |  F1={f1_r:.3f}')
        print(f'  Spatial : BA={ba_s:.3f} ± {std_s:.3f}  |  F1={f1_s:.3f}')
        print(f'  Inflation = {infl:+.2f} pp   |  elapsed = {(time.time()-t0)/60:.1f} min',
              flush=True)

        rows.append(dict(country=country, feature=feat_name, n_parcels=len(df),
                          n_classes=int(np.unique(y).size), n_blocks=int(n_blocks),
                          random_cv_ba=ba_r, random_cv_ba_std=std_r, random_cv_f1=f1_r,
                          spatial_cv_ba=ba_s, spatial_cv_ba_std=std_s, spatial_cv_f1=f1_s,
                          autocorrelation_inflation_pp=infl,
                          bb_excluded=True))   # marker so future re-runs see this is BB-clean
        done.add((country, feat_name))

        # 增量存 CSV
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f'  ✓ saved CSV ({len(rows)} combos done)', flush=True)

        del df, X, y, blocks
        gc.collect()

out = pd.DataFrame(rows)
print(f'\n=== ALL DONE: {len(rows)} combos ===')
out

## 7. Summary and figure

In [ ]:
print('=== Mean inflation across countries (per feature) ===')
for feat in FEATURES_TO_RUN:
    sub = out[out['feature'] == feat]
    if len(sub) == 0:
        continue
    print(f'  {feat:7s}: random = {sub["random_cv_ba"].mean():.3f}  '
          f'spatial = {sub["spatial_cv_ba"].mean():.3f}  '
          f'inflation = {sub["autocorrelation_inflation_pp"].mean():+.2f} pp')

import matplotlib.pyplot as plt

n_c = len(COUNTRIES_TO_RUN)
n_f = len(FEATURES_TO_RUN)
bar_w = 0.8 / n_f
x = np.arange(n_c)
colors = {'ae': '#1f77b4', 's2': '#ff7f0e', 'fusion': '#2ca02c'}

fig, ax = plt.subplots(figsize=(max(8, 1.4 * n_c), 5))
for i, feat in enumerate(FEATURES_TO_RUN):
    sub = out[out['feature'] == feat].set_index('country').reindex(COUNTRIES_TO_RUN)
    pos = x + (i - (n_f - 1) / 2) * bar_w
    ax.bar(pos, sub['random_cv_ba'], bar_w, color=colors[feat], alpha=0.45,
            label=f'{feat.upper()} random')
    ax.bar(pos, sub['spatial_cv_ba'], bar_w, color=colors[feat], alpha=1.0,
            edgecolor='black', linewidth=0.6, label=f'{feat.upper()} spatial')

ax.set_xticks(x); ax.set_xticklabels(COUNTRIES_TO_RUN)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Random vs spatial-block 5-fold CV per country (Exp 0 robustness)')
ax.legend(loc='lower right', fontsize=9, ncol=n_f)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig12_spatial_vs_random_cv.pdf', bbox_inches='tight')
fig.savefig(FIG_DIR / 'fig12_spatial_vs_random_cv.png', bbox_inches='tight', dpi=200)
plt.show()
print(f'\nSaved figure: {FIG_DIR / "fig12_spatial_vs_random_cv.pdf"}')

## 8. Interpretation

**What to expect / paper-relevant findings:**

- If **inflation < 3 pp** in most countries → spatial autocorrelation is small; the original random 5-fold numbers in Exp 0/1 are essentially correct, and a one-paragraph mention in the Limitations section suffices.
- If **inflation 3–5 pp** consistently → mention in a robustness paragraph in Section 5 (just after Exp 0) and update Table headline numbers as appropriate.
- If **inflation > 5 pp** in any country → that country's within-country accuracy is materially inflated; consider re-reporting Exp 0 with spatial-block CV as the headline.

**Key insight:** the *cross-border* (LOCO, pairwise) experiments are unaffected by within-country autocorrelation — they already evaluate on disjoint countries. So the central conclusions of the paper (Sub-RQs 1, 2, 3) are unchanged regardless of the inflation magnitude. This experiment is purely a robustness check on the Exp 0/1 baselines.